In [1]:
!mkdir -p ~/.kaggle

In [2]:
!cp kaggle.json ~/.kaggle/

In [3]:
!chmod 600 ~/.kaggle/kaggle.json

In [4]:
!kaggle datasets download -d uciml/pima-indians-diabetes-database

Dataset URL: https://www.kaggle.com/datasets/uciml/pima-indians-diabetes-database
License(s): CC0-1.0
100% 8.91k/8.91k [00:00<00:00, 20.8MB/s]



In [5]:
!unzip -q /content/pima-indians-diabetes-database.zip -d /content/data

In [17]:
!pip install keras-tuner

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 5.4 MB/s eta 0:00:00


In [18]:
import numpy as np
import pandas as pd
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Dropout
from tensorflow.keras.optimizers import Adam, SGD, RMSprop
import keras_tuner as kt

In [7]:
df = pd.read_csv('/content/data/diabetes.csv')
df.sample(5)

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
416,1,97,68,21,0,27.2,1.095,22,0
354,3,90,78,0,0,42.7,0.559,21,0
162,0,114,80,34,285,44.2,0.167,27,0
192,7,159,66,0,0,30.4,0.383,36,1
697,0,99,0,0,0,25.0,0.253,22,0


In [8]:
df.shape

(768, 9)

In [9]:
df.corr()['Outcome']

,Outcome
Pregnancies,0.221898
Glucose,0.466581
BloodPressure,0.065068
SkinThickness,0.074752
Insulin,0.130548
BMI,0.292695
DiabetesPedigreeFunction,0.173844
Age,0.238356
Outcome,1.000000


In [10]:
X = df.iloc[:,:-1].values
y = df.iloc[:,-1].values

In [11]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
X = sc.fit_transform(X)

In [12]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=1)

In [13]:
import tensorflow
from tensorflow import keras
from keras import Sequential
from keras.layers import Dense

In [64]:
model = Sequential()

model.add(Dense(32,activation='relu',input_dim=8))
model.add(Dense(1,activation='sigmoid'))

model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [65]:
early_stop = EarlyStopping(
    monitor='val_accuracy',
    patience=10,
    verbose=1,
    restore_best_weights=True
)

history = model.fit(
    X_train,
    y_train,
    epochs=100,
    batch_size=32,
    validation_data=(X_test, y_test),
    callbacks=[early_stop]
)

Epoch 1/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - accuracy: 0.6450 - loss: 0.6763 - val_accuracy: 0.6429 - val_loss: 0.6242
Epoch 2/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.6612 - loss: 0.6207 - val_accuracy: 0.6883 - val_loss: 0.5842
Epoch 3/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.6726 - loss: 0.5807 - val_accuracy: 0.7273 - val_loss: 0.5551
Epoch 4/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6971 - loss: 0.5518 - val_accuracy: 0.7532 - val_loss: 0.5339
Epoch 5/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7166 - loss: 0.5313 - val_accuracy: 0.7597 - val_loss: 0.5186
Epoch 6/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7329 - loss: 0.5149 - val_accuracy: 0.7857 - val_loss: 0.5075
Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7443 - loss: 0.5026 - val_accuracy: 0.7922 - val_loss: 0.5005
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7476 - loss: 0.4937 - val_accuracy: 0.8052 - 

val_accuracy: 0.8182

In [66]:
!pip install keras-tuner

In [67]:
import keras_tuner as kt

# 1. Finding best optimizer

In [68]:
def build_model(hp):

  model = Sequential()

  model.add(Dense(32,activation='relu',input_shape=(8,)))
  model.add(Dense(1,activation='sigmoid'))

  model.compile(optimizer=hp.Choice('optimizer', values=['adam', 'sgd', 'rmsprop', 'adagrad']),loss='binary_crossentropy',metrics=['accuracy'])

  return model

In [69]:
import shutil
shutil.rmtree('./untitled_project', ignore_errors=True)

tuner = kt.RandomSearch(build_model, objective='val_accuracy', max_trials=25)
tuner.search(X_train, y_train, epochs=20, validation_data=(X_test, y_test))

Trial 4 Complete [00h 00m 08s]
val_accuracy: 0.7857142686843872

Best val_accuracy So Far: 0.8051947951316833
Total elapsed time: 00h 00m 32s


In [70]:
tuner.results_summary()

Results summary
Results in ./untitled_project
Showing 10 best trials
Objective(name="val_accuracy", direction="max")

Trial 00 summary
Hyperparameters:
optimizer: adam
Score: 0.8051947951316833

Trial 03 summary
Hyperparameters:
optimizer: rmsprop
Score: 0.7857142686843872

Trial 01 summary
Hyperparameters:
optimizer: adagrad
Score: 0.7272727489471436

Trial 02 summary
Hyperparameters:
optimizer: sgd
Score: 0.7142857313156128


In [71]:
tuner.get_best_hyperparameters()[0].values

{'optimizer': 'adam'}

In [72]:
model = tuner.get_best_models(num_models=1)[0]
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 32)             │           288 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 321 (1.25 KB)

 Trainable params: 321 (1.25 KB)

 Non-trainable params: 0 (0.00 B)

In [73]:
early_stop = EarlyStopping(
    monitor='val_accuracy',
    patience=10,
    verbose=1,
    restore_best_weights=True
)

history = model.fit(
    X_train,
    y_train,
    epochs=100,
    batch_size=32,
    validation_data=(X_test, y_test),
    callbacks=[early_stop]
)

Epoch 1/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 26ms/step - accuracy: 0.7818 - loss: 0.4550 - val_accuracy: 0.7987 - val_loss: 0.4581
Epoch 2/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7850 - loss: 0.4525 - val_accuracy: 0.7987 - val_loss: 0.4570
Epoch 3/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7785 - loss: 0.4510 - val_accuracy: 0.7987 - val_loss: 0.4561
Epoch 4/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7769 - loss: 0.4492 - val_accuracy: 0.7987 - val_loss: 0.4560
Epoch 5/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7801 - loss: 0.4484 - val_accuracy: 0.8052 - val_loss: 0.4546
Epoch 6/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7818 - loss: 0.4475 - val_accuracy: 0.8052 - val_loss: 0.4534
Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7785 - loss: 0.4468 - val_accuracy: 0.8052 - val_loss: 0.4539
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7769 - loss: 0.4462 - val_accuracy: 0.8052 - 

val_accuracy: 0.8182

# 2. Finding best # neurons

In [74]:
def build_model(hp):

  model = Sequential()

  units = hp.Int('units', min_value=8, max_value=128, step=8)
  model.add(Dense(units=units,activation='relu',input_shape=(8,)))
  model.add(Dense(1,activation='sigmoid'))

  model.compile(optimizer='rmsprop',loss='binary_crossentropy',metrics=['accuracy'])

  return model

In [75]:
import shutil
shutil.rmtree('./untitled_project', ignore_errors=True)

tuner = kt.RandomSearch(build_model, objective='val_accuracy', max_trials=25)
tuner.search(X_train, y_train, epochs=20, validation_data=(X_test, y_test))

Trial 16 Complete [00h 00m 06s]
val_accuracy: 0.798701286315918

Best val_accuracy So Far: 0.8181818127632141
Total elapsed time: 00h 01m 43s


In [76]:
tuner.results_summary()

Results summary
Results in ./untitled_project
Showing 10 best trials
Objective(name="val_accuracy", direction="max")

Trial 05 summary
Hyperparameters:
units: 96
Score: 0.8181818127632141

Trial 06 summary
Hyperparameters:
units: 112
Score: 0.8181818127632141

Trial 11 summary
Hyperparameters:
units: 24
Score: 0.8181818127632141

Trial 03 summary
Hyperparameters:
units: 40
Score: 0.8116883039474487

Trial 07 summary
Hyperparameters:
units: 104
Score: 0.798701286315918

Trial 08 summary
Hyperparameters:
units: 120
Score: 0.798701286315918

Trial 10 summary
Hyperparameters:
units: 56
Score: 0.798701286315918

Trial 15 summary
Hyperparameters:
units: 80
Score: 0.798701286315918

Trial 00 summary
Hyperparameters:
units: 32
Score: 0.7922077775001526

Trial 01 summary
Hyperparameters:
units: 16
Score: 0.7922077775001526


In [81]:
model = tuner.get_best_models(num_models=1)[0]
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 96)             │           864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            97 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 961 (3.75 KB)

 Trainable params: 961 (3.75 KB)

 Non-trainable params: 0 (0.00 B)

In [82]:
early_stop = EarlyStopping(
    monitor='val_accuracy',
    patience=10,
    verbose=1,
    restore_best_weights=True
)

history = model.fit(
    X_train,
    y_train,
    epochs=100,
    batch_size=32,
    validation_data=(X_test, y_test),
    callbacks=[early_stop]
)

Epoch 1/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.7834 - loss: 0.4541 - val_accuracy: 0.8182 - val_loss: 0.4704
Epoch 2/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7834 - loss: 0.4494 - val_accuracy: 0.8117 - val_loss: 0.4708
Epoch 3/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7883 - loss: 0.4487 - val_accuracy: 0.8052 - val_loss: 0.4700
Epoch 4/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7850 - loss: 0.4472 - val_accuracy: 0.8117 - val_loss: 0.4688
Epoch 5/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7866 - loss: 0.4458 - val_accuracy: 0.8117 - val_loss: 0.4694
Epoch 6/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7866 - loss: 0.4450 - val_accuracy: 0.8052 - val_loss: 0.4669
Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7866 - loss: 0.4439 - val_accuracy: 0.8052 - val_loss: 0.4664
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7850 - loss: 0.4431 - val_accuracy: 0.8117 - 

val_accuracy: 0.8182

# 3. Finding # Layers

In [90]:
def build_model(hp):

  model = Sequential()

  model.add(Dense(96,activation='relu',input_dim=8))

  for i in range(hp.Int('units', min_value=1, max_value=10)):
    model.add(Dense(96,activation='relu'))

  model.add(Dense(1,activation='sigmoid'))

  model.compile(optimizer='rmsprop',loss='binary_crossentropy',metrics=['accuracy'])

  return model

In [91]:
import shutil
shutil.rmtree('./untitled_project', ignore_errors=True)

tuner = kt.RandomSearch(build_model, objective='val_accuracy', max_trials=25)
tuner.search(X_train, y_train, epochs=20, validation_data=(X_test, y_test))

Trial 10 Complete [00h 00m 08s]
val_accuracy: 0.8116883039474487

Best val_accuracy So Far: 0.8246753215789795
Total elapsed time: 00h 01m 20s


In [92]:
tuner.results_summary()

Results summary
Results in ./untitled_project
Showing 10 best trials
Objective(name="val_accuracy", direction="max")

Trial 03 summary
Hyperparameters:
units: 9
Score: 0.8246753215789795

Trial 06 summary
Hyperparameters:
units: 2
Score: 0.8181818127632141

Trial 07 summary
Hyperparameters:
units: 7
Score: 0.8181818127632141

Trial 09 summary
Hyperparameters:
units: 5
Score: 0.8116883039474487

Trial 01 summary
Hyperparameters:
units: 1
Score: 0.8051947951316833

Trial 02 summary
Hyperparameters:
units: 4
Score: 0.8051947951316833

Trial 05 summary
Hyperparameters:
units: 3
Score: 0.8051947951316833

Trial 00 summary
Hyperparameters:
units: 6
Score: 0.798701286315918

Trial 04 summary
Hyperparameters:
units: 10
Score: 0.798701286315918

Trial 08 summary
Hyperparameters:
units: 8
Score: 0.798701286315918


In [93]:
tuner.get_best_hyperparameters()[0].values

{'units': 9}

In [94]:
model = tuner.get_best_models(num_models=1)[0]
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 2 variables whereas the saved optimizer has 24 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 96)             │           864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 96)             │         9,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 96)             │         9,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 96)             │         9,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 96)             │         9,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 96)             │         9,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 96)             │         9,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 96)             │         9,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 96)             │         9,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 96)             │         9,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 1)              │            97 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 84,769 (331.13 KB)

 Trainable params: 84,769 (331.13 KB)

 Non-trainable params: 0 (0.00 B)

In [95]:
early_stop = EarlyStopping(
    monitor='val_accuracy',
    patience=10,
    verbose=1,
    restore_best_weights=True
)

history = model.fit(
    X_train,
    y_train,
    epochs=100,
    batch_size=32,
    validation_data=(X_test, y_test),
    callbacks=[early_stop]
)

Epoch 1/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 3s 36ms/step - accuracy: 0.7834 - loss: 0.4616 - val_accuracy: 0.7662 - val_loss: 0.4902
Epoch 2/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.8062 - loss: 0.4231 - val_accuracy: 0.7792 - val_loss: 0.5402
Epoch 3/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.8013 - loss: 0.4239 - val_accuracy: 0.8052 - val_loss: 0.5095
Epoch 4/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.8176 - loss: 0.4052 - val_accuracy: 0.7922 - val_loss: 0.5729
Epoch 5/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.8078 - loss: 0.4048 - val_accuracy: 0.7792 - val_loss: 0.5753
Epoch 6/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.8355 - loss: 0.3789 - val_accuracy: 0.8247 - val_loss: 0.4798
Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.8436 - loss: 0.3601 - val_accuracy: 0.8052 - val_loss: 0.6100
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.8453 - loss: 0.3535 - val_accuracy: 0.7987 -

val_accuracy: 0.8247

# 4. Finding everything

In [14]:
def build_model(hp):
    model = Sequential()

    for i in range(hp.Int('num_layers', min_value=1, max_value=5, step=1)):
        model.add(
            Dense(
                hp.Int('units_' + str(i), min_value=8, max_value=128, step=8),
                activation=hp.Choice('activation_' + str(i), values=['relu', 'sigmoid', 'tanh']),
                input_shape=(8,) if i == 0 else ()   # only on first layer
            )
        )
        model.add(
            Dropout(hp.Choice('dropout_' + str(i), values = [0.1,0.2,0.3,0.4,0.5,0.6]))
        )

    # Output layer for binary classification
    model.add(Dense(1, activation='sigmoid'))

    # Tune learning rate instead of just optimizer name
    optimizer_choice = hp.Choice('optimizer', values=['adam', 'sgd', 'rmsprop'])
    lr = hp.Float('learning_rate', min_value=1e-4, max_value=1e-2, sampling='log')

    if optimizer_choice == 'adam':
        optimizer = Adam(learning_rate=lr)
    elif optimizer_choice == 'sgd':
        optimizer = SGD(learning_rate=lr)
    else:
        optimizer = RMSprop(learning_rate=lr)

    model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])

    return model

In [133]:
tuner = kt.RandomSearch(
    build_model,
    objective='val_accuracy',
    max_trials=50,
    factor=3,               # how aggressively to eliminate bad trials
    directory='.',
    project_name='my_project2',
    overwrite=True
)
tuner.search(X_train, y_train, epochs=20, validation_data=(X_test, y_test))

Trial 25 Complete [00h 00m 08s]
val_accuracy: 0.8181818127632141

Best val_accuracy So Far: 0.8311688303947449
Total elapsed time: 00h 03m 39s


In [134]:
tuner.results_summary()

Results summary
Results in ./my_project2
Showing 10 best trials
Objective(name="val_accuracy", direction="max")

Trial 15 summary
Hyperparameters:
num_layers: 2
units_0: 24
activation_0: relu
dropout_0: 0.3
optimizer: adam
learning_rate: 0.0015538700676981138
units_1: 120
activation_1: relu
dropout_1: 0.3
units_2: 128
activation_2: sigmoid
dropout_2: 0.1
units_3: 24
activation_3: tanh
dropout_3: 0.1
units_4: 48
activation_4: tanh
dropout_4: 0.5
Score: 0.8311688303947449

Trial 19 summary
Hyperparameters:
num_layers: 4
units_0: 8
activation_0: relu
dropout_0: 0.6
optimizer: adam
learning_rate: 0.004184381429977421
units_1: 72
activation_1: tanh
dropout_1: 0.2
units_2: 48
activation_2: sigmoid
dropout_2: 0.3
units_3: 24
activation_3: relu
dropout_3: 0.1
units_4: 32
activation_4: sigmoid
dropout_4: 0.3
Score: 0.8246753215789795

Trial 04 summary
Hyperparameters:
num_layers: 4
units_0: 56
activation_0: relu
dropout_0: 0.6
optimizer: adam
learning_rate: 0.001187577274462532
units_1: 16
acti

In [135]:
# Clean retrain from scratch (more rigorous)
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
model = tuner.hypermodel.build(best_hps)  # fresh weights

In [136]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_2 (Dense)                 │ (None, 24)             │           216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 24)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 120)            │         3,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 120)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 1)              │           121 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,337 (13.04 KB)

 Trainable params: 3,337 (13.04 KB)

 Non-trainable params: 0 (0.00 B)

In [137]:
early_stop = EarlyStopping(
    monitor='val_accuracy',
    patience=10,
    verbose=1,
    restore_best_weights=True
)

history = model.fit(
    X_train,
    y_train,
    epochs=100,
    batch_size=32,
    validation_data=(X_test, y_test),
    callbacks=[early_stop]
)

Epoch 1/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - accuracy: 0.6466 - loss: 0.6186 - val_accuracy: 0.7078 - val_loss: 0.5647
Epoch 2/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.6971 - loss: 0.5687 - val_accuracy: 0.7597 - val_loss: 0.5190
Epoch 3/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7134 - loss: 0.5373 - val_accuracy: 0.7792 - val_loss: 0.4918
Epoch 4/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7182 - loss: 0.5180 - val_accuracy: 0.7987 - val_loss: 0.4845
Epoch 5/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7508 - loss: 0.4975 - val_accuracy: 0.7792 - val_loss: 0.4818
Epoch 6/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7296 - loss: 0.4795 - val_accuracy: 0.7792 - val_loss: 0.4796
Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7476 - loss: 0.4959 - val_accuracy: 0.7792 - val_loss: 0.4776
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7394 - loss: 0.4906 - val_accuracy: 0.7792 - 

val_accuracy: 0.8247

In [19]:
tuner = kt.Hyperband(
    build_model,
    objective='val_accuracy',
    max_epochs=50,          # ← Hyperband uses max_epochs, not epochs in search()
    factor=3,
    directory='.',
    project_name='my_project2',
    overwrite=True
)
tuner.search(X_train, y_train, validation_data=(X_test, y_test))

Trial 90 Complete [00h 00m 14s]
val_accuracy: 0.6623376607894897

Best val_accuracy So Far: 0.8441558480262756
Total elapsed time: 00h 13m 11s


In [28]:
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
model = tuner.hypermodel.build(best_hps)

In [29]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    verbose=1,
    restore_best_weights=True
)

history = model.fit(
    X_train, y_train,
    epochs=200,
    batch_size=32,
    validation_data=(X_test, y_test),
    callbacks=[early_stop]
)

Epoch 1/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 6s 140ms/step - accuracy: 0.5863 - loss: 0.6901 - val_accuracy: 0.7532 - val_loss: 0.5574
Epoch 2/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.7101 - loss: 0.5588 - val_accuracy: 0.8052 - val_loss: 0.4899
Epoch 3/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.7655 - loss: 0.5121 - val_accuracy: 0.7987 - val_loss: 0.4670
Epoch 4/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7720 - loss: 0.4845 - val_accuracy: 0.8052 - val_loss: 0.4579
Epoch 5/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7687 - loss: 0.4794 - val_accuracy: 0.8247 - val_loss: 0.4509
Epoch 6/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7736 - loss: 0.4843 - val_accuracy: 0.8312 - val_loss: 0.4525
Epoch 7/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7704 - loss: 0.4919 - val_accuracy: 0.8117 - val_loss: 0.4497
Epoch 8/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7785 - loss: 0.4808 - val_accuracy: 0.8117

val_accuracy: 0.818

In [25]:
tuner.results_summary()

Results summary
Results in ./my_project2
Showing 10 best trials
Objective(name="val_accuracy", direction="max")

Trial 0072 summary
Hyperparameters:
num_layers: 2
units_0: 72
activation_0: relu
dropout_0: 0.3
optimizer: adam
learning_rate: 0.0019470721462503994
units_1: 8
activation_1: tanh
dropout_1: 0.2
units_2: 72
activation_2: tanh
dropout_2: 0.2
units_3: 112
activation_3: sigmoid
dropout_3: 0.1
units_4: 128
activation_4: tanh
dropout_4: 0.3
tuner/epochs: 50
tuner/initial_epoch: 17
tuner/bracket: 2
tuner/round: 2
tuner/trial_id: 0067
Score: 0.8441558480262756

Trial 0051 summary
Hyperparameters:
num_layers: 4
units_0: 16
activation_0: relu
dropout_0: 0.2
optimizer: adam
learning_rate: 0.005486272562976355
units_1: 120
activation_1: relu
dropout_1: 0.3
units_2: 104
activation_2: tanh
dropout_2: 0.5
units_3: 16
activation_3: tanh
dropout_3: 0.5
tuner/epochs: 50
tuner/initial_epoch: 17
tuner/bracket: 3
tuner/round: 3
tuner/trial_id: 0048
units_4: 72
activation_4: sigmoid
dropout_4: 0.